# Notebook 6 — Simulación Monte Carlo del cuadro completo (modelo pre-Mundial)

**Objetivo**: a diferencia del walk-forward del Notebook 4 (que ya incorpora la fase de
grupos real antes de evaluar la primera ronda de eliminatoria), este notebook congela el
modelo **tal como estaba entrenado el día antes de que arrancara el Mundial** — sin haber
visto ni un solo resultado del torneo, ni siquiera la fase de grupos — y simula con él
**todo el cuadro de eliminatoria de un tirón**, de dieciseisavos a la final.

Como el resultado de cada partido es incierto, no basta con propagar "quién es más
probable que gane" ronda a ronda (eso solo daría un único camino, ignorando la
incertidumbre acumulada): se simula el marcador de cada partido miles de veces siguiendo
las distribuciones de Poisson que da el modelo, y se cuenta con qué frecuencia llega cada
selección a octavos / cuartos / semis / final / campeón — el enfoque estándar de este
tipo de predictores de torneo (estilo FiveThirtyEight).

Entrada: `data/processed/partidos_features.csv` (Notebook 2), `data/raw/wc2026_calendario.json`
(estructura real del cuadro, Notebook 1 sección 1.5).
Salida: `data/processed/simulacion_probabilidades_equipos.csv`,
`data/processed/simulacion_dieciseisavos.csv`.

In [1]:
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DIR_RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_PROCESSED = DIR_RAIZ / "data" / "processed"
DIR_RAW = DIR_RAIZ / "data" / "raw"

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

SEMILLA = 42
MAX_GOLES_MATRIZ = 8  # igual que en el Notebook 4: la matriz se normaliza, pero conviene cubrir bien lambdas altas
N_SIMULACIONES = 3000

FECHA_CORTE_HISTORICO = pd.Timestamp("1990-01-01")
FECHA_INICIO_MUNDIAL = pd.Timestamp("2026-06-11")
FEATURES_MODELO = [
    "elo_diff", "tendencia_elo_local", "tendencia_elo_visitante",
    "dif_forma_gf_5", "dif_forma_gf_10", "dif_racha_5", "dif_racha_10",
    "dias_descanso_local", "dias_descanso_visitante",
]

df = pd.read_csv(DIR_PROCESSED / "partidos_features.csv", parse_dates=["fecha"])
df = df[df["fecha"] >= FECHA_CORTE_HISTORICO].reset_index(drop=True)
print(f"Partidos desde {FECHA_CORTE_HISTORICO.date()}: {len(df):,}")

Partidos desde 1990-01-01: 32,380


## 6.1 Modelo congelado pre-Mundial

Se reentrena el Poisson GLM (la familia ganadora en el Notebook 4) usando **solo**
partidos anteriores al Mundial — a propósito, sin incorporar la fase de grupos, para que
la simulación represente de verdad "qué habría dicho el modelo el día antes de empezar
el torneo".

In [2]:
def entrenar_poisson_glm(X: pd.DataFrame, y: pd.Series, alpha: float = 0.5):
    return make_pipeline(StandardScaler(), PoissonRegressor(alpha=alpha, max_iter=1000)).fit(X, y)


df_train_inicial = df[(df["fecha"] < FECHA_INICIO_MUNDIAL) & df["jugado"]].copy()
X_train_inicial = df_train_inicial[FEATURES_MODELO]

# El Notebook 4 ya guarda este mismo modelo (GLM entrenado solo con datos
# pre-Mundial) como checkpoint "pre_mundial" -- si existe y coincide en
# familia, se reutiliza en vez de reentrenar desde cero.
RUTA_CHECKPOINT_PRE_MUNDIAL = DIR_RAIZ / "models" / "checkpoints" / "pre_mundial"
metadata_checkpoint = RUTA_CHECKPOINT_PRE_MUNDIAL / "metadata.json"

if metadata_checkpoint.exists() and json.loads(metadata_checkpoint.read_text())["familia"] == "glm":
    import joblib
    modelo_local_pre_mundial = joblib.load(RUTA_CHECKPOINT_PRE_MUNDIAL / "modelo_goles_local.joblib")
    modelo_visitante_pre_mundial = joblib.load(RUTA_CHECKPOINT_PRE_MUNDIAL / "modelo_goles_visitante.joblib")
    print(f"Modelo congelado cargado desde el checkpoint '{RUTA_CHECKPOINT_PRE_MUNDIAL}' "
          f"(Notebook 4), sin reentrenar.")
else:
    modelo_local_pre_mundial = entrenar_poisson_glm(X_train_inicial, df_train_inicial["goles_local"])
    modelo_visitante_pre_mundial = entrenar_poisson_glm(X_train_inicial, df_train_inicial["goles_visitante"])
    print(f"Modelo congelado entrenado con {len(df_train_inicial):,} partidos, hasta "
          f"{df_train_inicial['fecha'].max().date()} (ni un partido de 2026 incluido).")


def log_verosimilitud_dixon_coles(rho: float, lam: np.ndarray, mu: np.ndarray, x: np.ndarray, y: np.ndarray) -> float:
    '''Misma corrección que en el Notebook 4 (sección 4.5) -- se recalibra
    aquí porque este notebook se ejecuta de forma independiente.'''
    tau = np.ones_like(lam)
    es_0_0 = (x == 0) & (y == 0)
    es_0_1 = (x == 0) & (y == 1)
    es_1_0 = (x == 1) & (y == 0)
    es_1_1 = (x == 1) & (y == 1)
    tau[es_0_0] = 1 - lam[es_0_0] * mu[es_0_0] * rho
    tau[es_0_1] = 1 + lam[es_0_1] * rho
    tau[es_1_0] = 1 + mu[es_1_0] * rho
    tau[es_1_1] = 1 - rho
    tau = np.clip(tau, 1e-8, None)
    return float((np.log(tau) + poisson.logpmf(x, lam) + poisson.logpmf(y, mu)).sum())


def ajustar_rho_dixon_coles(modelo_local, modelo_visitante, X: pd.DataFrame,
                             goles_local: pd.Series, goles_visitante: pd.Series) -> float:
    lam = np.clip(modelo_local.predict(X), 0.01, None)
    mu = np.clip(modelo_visitante.predict(X), 0.01, None)
    x, y = goles_local.to_numpy(), goles_visitante.to_numpy()
    candidatos_rho = np.linspace(-0.3, 0.3, 121)
    log_verosimilitudes = [log_verosimilitud_dixon_coles(r, lam, mu, x, y) for r in candidatos_rho]
    return float(candidatos_rho[np.argmax(log_verosimilitudes)])


RHO_DIXON_COLES = ajustar_rho_dixon_coles(modelo_local_pre_mundial, modelo_visitante_pre_mundial,
                                            X_train_inicial, df_train_inicial["goles_local"], df_train_inicial["goles_visitante"])
print(f"Corrección de Dixon-Coles: rho = {RHO_DIXON_COLES:.4f}")

Modelo congelado cargado desde el checkpoint '/Users/danielcanteragomez/portfolio/wc-2026-match-predictor/models/checkpoints/pre_mundial' (Notebook 4), sin reentrenar.


Corrección de Dixon-Coles: rho = -0.0350


## 6.2 Reconstruir el cuadro de eliminatoria como grafo de dependencias

El JSON ya resuelve un partido futuro a un nombre real de selección en cuanto se conoce
en la realidad (p.ej. octavos ya dice `"Paraguay"` en vez de `"W74"` porque el
dieciseisavo que lo decidía ya se jugó) — pero esta simulación debe ignorar esa
información real y decidirlo ella misma. Por eso se **renormalizan** los octavos en
adelante a referencias `"W<num>"` puras, buscando en qué dieciseisavo (73-88) quedó cada
selección como ganadora real (usando penaltis/prórroga si aplica, no solo el marcador de
90 minutos) — así la simulación parte solo del dibujo del cuadro, nunca de un resultado.

In [3]:
RUTA_CALENDARIO_JSON = DIR_RAW / "wc2026_calendario.json"
MAPEO_NOMBRES_JSON = {
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "USA": "United States",
}

with open(RUTA_CALENDARIO_JSON, encoding="utf-8") as f:
    calendario = json.load(f)["matches"]

partidos_eliminatoria_json = {m["num"]: m for m in calendario if "num" in m}


def ganador_real(partido: dict) -> str | None:
    """Quién avanzó de verdad: penaltis > prórroga > 90 minutos, en ese orden
    de prioridad (si hay tanda de penaltis, el marcador de 90' pudo ir empatado)."""
    marcador = partido.get("score")
    if marcador is None:
        return None
    for clave in ("p", "et", "ft"):
        if clave in marcador:
            g1, g2 = marcador[clave]
            if g1 != g2:
                return partido["team1"] if g1 > g2 else partido["team2"]
    return None


# selección -> número de dieciseisavo (73-88) del que salió como ganadora real.
equipo_a_num_dieciseisavo = {}
for num, partido in partidos_eliminatoria_json.items():
    if num > 88:
        continue
    ganador = ganador_real(partido)
    if ganador:
        equipo_a_num_dieciseisavo[MAPEO_NOMBRES_JSON.get(ganador, ganador)] = num


def normalizar_referencia(referencia: str) -> str:
    equipo = MAPEO_NOMBRES_JSON.get(referencia, referencia)
    return f"W{equipo_a_num_dieciseisavo[equipo]}" if equipo in equipo_a_num_dieciseisavo else equipo


bracket = {}
for num, partido in partidos_eliminatoria_json.items():
    if num <= 88:
        referencia_a = MAPEO_NOMBRES_JSON.get(partido["team1"], partido["team1"])
        referencia_b = MAPEO_NOMBRES_JSON.get(partido["team2"], partido["team2"])
    else:
        referencia_a = normalizar_referencia(partido["team1"])
        referencia_b = normalizar_referencia(partido["team2"])
    bracket[num] = {"fecha": pd.Timestamp(partido["date"]), "referencia_a": referencia_a, "referencia_b": referencia_b}

bracket_ordenado = sorted(bracket.items())  # por num ascendente: toda dependencia tiene num menor
for num, partido in bracket_ordenado:
    print(f"  {num}: {partido['referencia_a']} vs {partido['referencia_b']} ({partido['fecha'].date()})")

  73: South Africa vs Canada (2026-06-28)
  74: Germany vs Paraguay (2026-06-29)
  75: Netherlands vs Morocco (2026-06-29)
  76: Brazil vs Japan (2026-06-29)
  77: France vs Sweden (2026-06-30)
  78: Ivory Coast vs Norway (2026-06-30)
  79: Mexico vs Ecuador (2026-06-30)
  80: England vs DR Congo (2026-07-01)
  81: United States vs Bosnia and Herzegovina (2026-07-01)
  82: Belgium vs Senegal (2026-07-01)
  83: Portugal vs Croatia (2026-07-02)
  84: Spain vs Austria (2026-07-02)
  85: Switzerland vs Algeria (2026-07-02)
  86: Argentina vs Cape Verde (2026-07-03)
  87: Colombia vs Ghana (2026-07-03)
  88: Australia vs Egypt (2026-07-03)
  89: W74 vs W77 (2026-07-04)
  90: W73 vs W75 (2026-07-04)
  91: W76 vs W78 (2026-07-05)
  92: W79 vs W80 (2026-07-05)
  93: W83 vs W84 (2026-07-06)
  94: W81 vs W82 (2026-07-06)
  95: W86 vs W88 (2026-07-07)
  96: W85 vs W87 (2026-07-07)
  97: W89 vs W90 (2026-07-09)
  98: W93 vs W94 (2026-07-10)
  99: W91 vs W92 (2026-07-11)
  100: W95 vs W96 (2026-07-

## 6.3 Snapshot de cada selección, congelado antes del Mundial

Las features del modelo (`elo_diff`, `dif_forma_gf_5`, ...) son diferencias entre dos
selecciones, no cantidades propias de una sola — para poder construir la feature de
CUALQUIER cruce hipotético (p.ej. un cuarto de final entre dos selecciones que en la
realidad no llegaron a cruzarse) hace falta primero descomponerlas en la cantidad propia
de cada selección (su Elo, su tendencia, su forma) tal como estaba en su última fila
antes del Mundial, y recombinarlas después para cada pareja.

In [4]:
def snapshot_pre_mundial(equipo: str, df_train: pd.DataFrame) -> dict:
    es_local = df_train["equipo_local"] == equipo
    es_visitante = df_train["equipo_visitante"] == equipo
    ultima_local = df_train.loc[es_local, "fecha"].max() if es_local.any() else pd.Timestamp.min
    ultima_visitante = df_train.loc[es_visitante, "fecha"].max() if es_visitante.any() else pd.Timestamp.min

    if ultima_local >= ultima_visitante:
        fila = df_train.loc[es_local].sort_values("fecha").iloc[-1]
        sufijo = "_local"
    else:
        fila = df_train.loc[es_visitante].sort_values("fecha").iloc[-1]
        sufijo = "_visitante"

    return {
        "elo": fila[f"elo_actual{sufijo}"],
        "tendencia_elo": fila[f"tendencia_elo{sufijo}"],
        "forma_gf_5": fila[f"prom_gf_5{sufijo}"],
        "forma_gf_10": fila[f"prom_gf_10{sufijo}"],
        "racha_5": fila[f"racha_puntos_5{sufijo}"],
        "racha_10": fila[f"racha_puntos_10{sufijo}"],
        "ultima_fecha": max(ultima_local, ultima_visitante),
    }


equipos_mundial_2026 = sorted({partido["referencia_a"] for _, partido in bracket_ordenado if not re.match(r"^[WL]\d+$", partido["referencia_a"])}
                                | {partido["referencia_b"] for _, partido in bracket_ordenado if not re.match(r"^[WL]\d+$", partido["referencia_b"])})

snapshots = {equipo: snapshot_pre_mundial(equipo, df_train_inicial) for equipo in equipos_mundial_2026}
print(f"Snapshot pre-Mundial calculado para {len(snapshots)} selecciones (las 32 de dieciseisavos).")

Snapshot pre-Mundial calculado para 32 selecciones (las 32 de dieciseisavos).


## 6.4 Predecir un cruce (real o hipotético) y simular su marcador

`dias_descanso` sí varía de una simulación a otra (depende de qué ronda jugó cada
selección por última vez EN ESA simulación, no de la realidad) — el resto de features
quedan congeladas en el snapshot pre-Mundial. Se cachea la predicción por
`(equipo_a, equipo_b, fecha, descanso_a, descanso_b)` porque los dieciseisavos son el
mismo cruce en las `N_SIMULACIONES` simulaciones — sin caché se recalcularía la misma
predicción miles de veces para nada.

In [5]:
def construir_features_hipotetico(equipo_a: str, equipo_b: str, fecha: pd.Timestamp,
                                    ultimo_partido: dict, snapshots: dict) -> pd.DataFrame:
    snap_a, snap_b = snapshots[equipo_a], snapshots[equipo_b]
    return pd.DataFrame([{
        "elo_diff": snap_a["elo"] - snap_b["elo"],
        "tendencia_elo_local": snap_a["tendencia_elo"],
        "tendencia_elo_visitante": snap_b["tendencia_elo"],
        "dif_forma_gf_5": snap_a["forma_gf_5"] - snap_b["forma_gf_5"],
        "dif_forma_gf_10": snap_a["forma_gf_10"] - snap_b["forma_gf_10"],
        "dif_racha_5": snap_a["racha_5"] - snap_b["racha_5"],
        "dif_racha_10": snap_a["racha_10"] - snap_b["racha_10"],
        "dias_descanso_local": (fecha - ultimo_partido[equipo_a]).days,
        "dias_descanso_visitante": (fecha - ultimo_partido[equipo_b]).days,
    }])[FEATURES_MODELO]


cache_lambdas: dict = {}


def predecir_lambda(equipo_a: str, equipo_b: str, fecha: pd.Timestamp, ultimo_partido: dict) -> tuple[float, float]:
    clave = (equipo_a, equipo_b, fecha, ultimo_partido[equipo_a], ultimo_partido[equipo_b])
    if clave not in cache_lambdas:
        X = construir_features_hipotetico(equipo_a, equipo_b, fecha, ultimo_partido, snapshots)
        lam_a = float(np.clip(modelo_local_pre_mundial.predict(X), 0.01, None)[0])
        lam_b = float(np.clip(modelo_visitante_pre_mundial.predict(X), 0.01, None)[0])
        cache_lambdas[clave] = (lam_a, lam_b)
    return cache_lambdas[clave]


def matriz_probabilidad_conjunta(lam_a: float, lam_b: float, max_goles: int = MAX_GOLES_MATRIZ) -> np.ndarray:
    '''Con la corrección de Dixon-Coles en los 4 marcadores bajos (mismo rho
    calibrado en 6.1), igual que en el Notebook 4.'''
    goles = np.arange(max_goles + 1)
    matriz = np.outer(poisson.pmf(goles, lam_a), poisson.pmf(goles, lam_b))
    matriz[0, 0] *= 1 - lam_a * lam_b * RHO_DIXON_COLES
    matriz[0, 1] *= 1 + lam_a * RHO_DIXON_COLES
    matriz[1, 0] *= 1 + lam_b * RHO_DIXON_COLES
    matriz[1, 1] *= 1 - RHO_DIXON_COLES
    matriz = np.clip(matriz, 0, None)
    return matriz / matriz.sum()


PATRON_PLACEHOLDER = re.compile(r"^([WL])(\d+)$")


def resolver_equipo(referencia: str, ganadores: dict, perdedores: dict) -> str:
    m = PATRON_PLACEHOLDER.match(referencia)
    if m is None:
        return referencia
    tipo, num = m.group(1), int(m.group(2))
    return ganadores[num] if tipo == "W" else perdedores[num]

## 6.5 Una simulación completa del cuadro

Un empate simulado a 90 minutos se decide con un sorteo ponderado por la matriz de
probabilidad conjunta (no con el criterio determinista de `resolver_eliminatoria` del
Notebook 4) — aquí interesa precisamente que un cruce parejo pueda caer para cualquiera
de los dos lados según a quién le toque en cada simulación, para que la incertidumbre se
propague de verdad de ronda en ronda.

In [6]:
def simular_torneo_una_vez(rng: np.random.Generator) -> tuple[dict, dict]:
    ultimo_partido = {equipo: snap["ultima_fecha"] for equipo, snap in snapshots.items()}
    ganadores, perdedores, marcadores = {}, {}, {}

    for num, partido in bracket_ordenado:
        equipo_a = resolver_equipo(partido["referencia_a"], ganadores, perdedores)
        equipo_b = resolver_equipo(partido["referencia_b"], ganadores, perdedores)
        fecha = partido["fecha"]

        lam_a, lam_b = predecir_lambda(equipo_a, equipo_b, fecha, ultimo_partido)
        goles_a, goles_b = int(rng.poisson(lam_a)), int(rng.poisson(lam_b))

        if goles_a != goles_b:
            ganador = equipo_a if goles_a > goles_b else equipo_b
        else:
            matriz = matriz_probabilidad_conjunta(lam_a, lam_b)
            prob_a = np.tril(matriz, -1).sum()
            prob_b = np.triu(matriz, 1).sum()
            ganador = equipo_a if rng.random() < prob_a / (prob_a + prob_b) else equipo_b

        perdedores[num] = equipo_b if ganador == equipo_a else equipo_a
        ganadores[num] = ganador
        marcadores[num] = {"equipo_a": equipo_a, "equipo_b": equipo_b, "lam_a": lam_a, "lam_b": lam_b,
                            "goles_a": goles_a, "goles_b": goles_b, "ganador": ganador}
        ultimo_partido[equipo_a] = fecha
        ultimo_partido[equipo_b] = fecha

    return ganadores, marcadores


rng_ejemplo = np.random.default_rng(SEMILLA)
ganadores_ejemplo, marcadores_ejemplo = simular_torneo_una_vez(rng_ejemplo)
print("Campeón de esta simulación de ejemplo:", ganadores_ejemplo[104])

Campeón de esta simulación de ejemplo: Germany


## 6.6 Monte Carlo: repetir la simulación miles de veces

Se cuenta, para cada selección, cuántas de las `N_SIMULACIONES` veces gana su partido de
cada ronda (= llega a la siguiente) — la tabla estándar de probabilidades de un
predictor de torneo. Los dieciseisavos son el único bloque con el cruce fijo en las
`N_SIMULACIONES` (los octavos en adelante dependen de quién gane antes), así que ahí
además se guarda el marcador previsto (determinista, viene del modelo congelado) junto a
la probabilidad de que cada lado lo gane (esa sí variable, por el sorteo de Poisson).

In [7]:
ETAPAS = [
    ("alcanza_octavos", range(73, 89)),
    ("alcanza_cuartos", range(89, 97)),
    ("alcanza_semis", range(97, 101)),
    ("alcanza_final", range(101, 103)),
    ("campeon", range(104, 105)),
]

conteo_etapas = defaultdict(lambda: defaultdict(int))
conteo_gana_dieciseisavo = defaultdict(int)  # num (73-88) -> veces que gana equipo_a

rng = np.random.default_rng(SEMILLA)
for _ in range(N_SIMULACIONES):
    ganadores, marcadores = simular_torneo_una_vez(rng)
    for nombre_etapa, rango in ETAPAS:
        for num in rango:
            conteo_etapas[ganadores[num]][nombre_etapa] += 1
    for num in range(73, 89):
        if marcadores[num]["ganador"] == marcadores[num]["equipo_a"]:
            conteo_gana_dieciseisavo[num] += 1

print(f"{N_SIMULACIONES:,} simulaciones completas del cuadro.")

3,000 simulaciones completas del cuadro.


In [8]:
filas_probabilidades = []
for equipo in equipos_mundial_2026:
    fila = {"seleccion": equipo}
    for nombre_etapa, _ in ETAPAS:
        fila[nombre_etapa] = conteo_etapas[equipo][nombre_etapa] / N_SIMULACIONES
    filas_probabilidades.append(fila)

df_probabilidades = pd.DataFrame(filas_probabilidades).sort_values("campeon", ascending=False).reset_index(drop=True)
print("=== Probabilidad de cada selección de alcanzar cada ronda (modelo pre-Mundial) ===")
print(df_probabilidades.round(4).to_string(index=False))

ruta_probabilidades = DIR_PROCESSED / "simulacion_probabilidades_equipos.csv"
df_probabilidades.to_csv(ruta_probabilidades, index=False)
print(f"\nGuardado: {ruta_probabilidades}")

=== Probabilidad de cada selección de alcanzar cada ronda (modelo pre-Mundial) ===
             seleccion  alcanza_octavos  alcanza_cuartos  alcanza_semis  alcanza_final  campeon
               Germany           0.9230           0.5833         0.4687         0.3263   0.2150
             Argentina           0.9530           0.8863         0.7587         0.4517   0.2047
                France           0.9510           0.3903         0.3117         0.2267   0.1527
                 Spain           0.7400           0.3283         0.2530         0.0997   0.0707
              Portugal           0.6997           0.4187         0.2870         0.0933   0.0583
                Brazil           0.6780           0.4620         0.3253         0.1613   0.0517
               Belgium           0.8157           0.5727         0.2190         0.0750   0.0473
                Norway           0.5420           0.2230         0.1583         0.0897   0.0303
           Netherlands           0.6017           0.3

In [9]:
filas_dieciseisavos = []
for num in range(73, 89):
    partido = bracket[num]
    equipo_a, equipo_b = partido["referencia_a"], partido["referencia_b"]
    lam_a, lam_b = cache_lambdas[(equipo_a, equipo_b, partido["fecha"], snapshots[equipo_a]["ultima_fecha"], snapshots[equipo_b]["ultima_fecha"])]
    prob_a = conteo_gana_dieciseisavo[num] / N_SIMULACIONES
    # floor(lambda), no round(lambda): es la moda real de una Poisson.
    filas_dieciseisavos.append({
        "num_partido": num,
        "fecha": partido["fecha"],
        "equipo_a": equipo_a,
        "equipo_b": equipo_b,
        "marcador_previsto": f"{int(np.floor(lam_a))}-{int(np.floor(lam_b))}",
        "prob_gana_a": prob_a,
        "prob_gana_b": 1 - prob_a,
    })

df_dieciseisavos = pd.DataFrame(filas_dieciseisavos)
print("=== Dieciseisavos: marcador previsto y probabilidad, modelo pre-Mundial ===")
print(df_dieciseisavos.round(4).to_string(index=False))

ruta_dieciseisavos = DIR_PROCESSED / "simulacion_dieciseisavos.csv"
df_dieciseisavos.to_csv(ruta_dieciseisavos, index=False)
print(f"\nGuardado: {ruta_dieciseisavos}")

=== Dieciseisavos: marcador previsto y probabilidad, modelo pre-Mundial ===
 num_partido      fecha      equipo_a               equipo_b marcador_previsto  prob_gana_a  prob_gana_b
          73 2026-06-28  South Africa                 Canada               1-1       0.4463       0.5537
          74 2026-06-29       Germany               Paraguay               2-0       0.9230       0.0770
          75 2026-06-29   Netherlands                Morocco               1-1       0.6017       0.3983
          76 2026-06-29        Brazil                  Japan               1-1       0.6780       0.3220
          77 2026-06-30        France                 Sweden               2-0       0.9510       0.0490
          78 2026-06-30   Ivory Coast                 Norway               1-1       0.4580       0.5420
          79 2026-06-30        Mexico                Ecuador               1-0       0.6733       0.3267
          80 2026-07-01       England               DR Congo               2-0      

## Resumen

- El modelo congelado el día antes del Mundial (sin ver ni la fase de grupos) simula
  `N_SIMULACIONES` veces el cuadro completo de eliminatoria, propagando la incertidumbre
  de cada partido con sorteos de Poisson en vez de asumir siempre "gana el favorito".
- `data/processed/simulacion_probabilidades_equipos.csv`: probabilidad de cada selección
  de alcanzar octavos/cuartos/semis/final/ser campeón — la vista "día 0" del torneo,
  comparable contra lo que de verdad pasó (Notebook 4) para ver cuánto acertó o falló el
  modelo antes de que se jugara ni un partido.
- `data/processed/simulacion_dieciseisavos.csv`: marcador previsto y probabilidad de cada
  cruce de dieciseisavos (el único bloque con cruces fijos, todos conocidos de antemano).